
# Photovoltaic Data Preparation and Validation

## Objective

Prepare and validate photovoltaic installation data for six departments in southeastern France (04, 05, 07, 26, 38, and 84) over the 2017–2025 period.

## Methodology

- Clean and standardize the raw datasets.
- Check data consistency and document validation procedures.
- Build a departmental panel and generate summary tables by year.
- Analyze empirical energy yields and assess the robustness of observed trends.
- Add departmental land areas and calculate photovoltaic capacity and estimated energy yield per square kilometer.

## Outputs

- Cleaned installation-level dataset.
- Departmental summary by year.
- Reference energy yields for comparison with theoretical PVGIS estimates.

## Limitations

Observed energy yields may be affected by weather conditions, measurement issues, changes in the installation sample, and operating constraints. Reference yields are empirical estimates and should not be interpreted as purely theoretical solar potential.


In [10]:
import pandas as pd

In [11]:
import pandas as pd

df = pd.read_csv("dataset\\data raw\\2026-installation-production-stockage-electricite-agrege.csv", sep=";", encoding="utf-8", low_memory=False,
                 dtype={"codeDepartement": str, "codeINSEECommune": str, "codeIRIS": str})

# 1. Quelles valeurs pour la filière et la technologie ?
print(df["filiere"].value_counts(dropna=False))
print(df[df["filiere"].str.contains("olaire", case=False, na=False)]
        ["technologie"].value_counts(dropna=False))

# 2. Filtre sur tes départements
deps = ["26", "07", "38", "05", "04", "84"]
sol = df[df["codeDepartement"].isin(deps) &
         df["filiere"].str.contains("olaire", case=False, na=False)].copy()
print(sol.shape)
print(sol.groupby("codeDepartement").size())

# 3. Structure des lignes et puissance
print(sol["nbInstallations"].value_counts().head(10))
print(sol["puisMaxInstallee"].describe())

# 4. Taux de remplissage des colonnes clés
cols = ["dateRaccordement", "dateMiseEnService", "dateDeraccordement",
        "energieAnnuelleGlissanteProduite", "energieAnnuelleGlissanteInjectee"]
print(sol[cols].notna().mean().round(3))

# 5. Doublons éventuels (versions multiples d'une même installation)
print(sol["codeEICResourceObject"].duplicated().sum())
print(sol["gestionnaire"].value_counts())

filiere
Solaire                       132856
Hydraulique                     2752
Eolien                          2458
Bioénergies                     1259
Thermique non renouvelable      1238
Stockage non hydraulique         922
Nucléaire                         57
Autre                             41
Energies Marines                   4
Géothermie                         2
Name: count, dtype: int64
technologie
Photovoltaïque     128273
NaN                  3372
Autre                1205
Thermodynamique         5
Batterie                1
Name: count, dtype: int64
(8736, 50)
codeDepartement
04    1023
05    1004
07    1066
26    2143
38    2128
84    1372
dtype: int64
nbInstallations
1     6332
11     106
10     101
13     100
14      96
18      88
12      83
16      77
19      71
20      67
Name: count, dtype: int64
count     8736.000000
mean       290.038972
std        901.016248
min         27.100000
25%         90.000000
50%        113.070000
75%        220.000000
max      16650.0

In [12]:
sol = df[df["codeDepartement"].isin(deps) & (df["filiere"] == "Solaire")].copy()
sol["agrege"] = sol["nbInstallations"] > 1

# A. Les lignes agrégées sont-elles celles sans EIC ni gestionnaire ?
print(pd.crosstab(sol["agrege"], sol["codeEICResourceObject"].isna()))
print(pd.crosstab(sol["agrege"], sol["gestionnaire"].isna()))

# B. Technologie dans ton périmètre
print(pd.crosstab(sol["technologie"].fillna("NaN"), sol["agrege"]))

# C. Seuil 36 kW sur les lignes individuelles
ind = sol[~sol["agrege"]].copy()
print((ind["puisMaxInstallee"] < 36).sum(), (ind["puisMaxRac"] < 36).sum())
print(ind[["puisMaxInstallee", "puisMaxRac"]].describe())

# D. Dates
cols_date = [c for c in sol.columns if c.lower().startswith("datemiseenservice")] + ["dateRaccordement"]
print(sol[cols_date].notna().mean())
for c in ["dateRaccordement", "dateMiseEnService"]:
    sol[c] = pd.to_datetime(sol[c], errors="coerce")
ind = sol[~sol["agrege"]].copy()
print(sol.groupby("agrege")["dateRaccordement"].describe())       # dates des lignes agrégées crédibles ?
print(ind["dateRaccordement"].dt.year.value_counts().sort_index())

# E. Unité de la production injectée
ind["kwh_par_kw"] = ind["energieAnnuelleGlissanteInjectee"] / ind["puisMaxInstallee"]
print(ind["kwh_par_kw"].describe())
print(ind.groupby("codeDepartement")[["puisMaxInstallee", "energieAnnuelleGlissanteInjectee"]].sum())
print(sol.groupby("agrege")["puisMaxInstallee"].sum())            # poids des petites installations

codeEICResourceObject  False  True 
agrege                             
False                   6332      0
True                       0   2404
gestionnaire  False  True 
agrege                    
False          6332      0
True              0   2404
agrege           False  True 
technologie                  
NaN                  9      2
Photovoltaïque    6322   2402
Thermodynamique      1      0
0 0
       puisMaxInstallee    puisMaxRac
count       6332.000000   6193.000000
mean         315.797310    314.146442
std         1038.128751   1048.661239
min           36.960000     37.000000
25%           90.000000     90.000000
50%          108.000000    100.000000
75%          220.000000    220.000000
max        16650.000000  16000.000000
dateMiseEnService                  0.999657
dateMiseEnservice (format date)    0.999657
dateRaccordement                   1.000000
dtype: float64
       count                           mean                  min  \
agrege                               

C:\Users\willi\AppData\Local\Temp\ipykernel_18608\1678582292.py:20: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sol[c] = pd.to_datetime(sol[c], errors="coerce")
C:\Users\willi\AppData\Local\Temp\ipykernel_18608\1678582292.py:20: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sol[c] = pd.to_datetime(sol[c], errors="coerce")


In [13]:
fmt = "%d/%m/%Y"   # format explicite : supprime l'avertissement et évite les erreurs jour/mois
for c in ["dateRaccordement", "dateMiseEnService"]:
    sol[c] = pd.to_datetime(sol[c], format=fmt, errors="coerce")

ind = sol[(~sol["agrege"]) & (sol["technologie"] == "Photovoltaïque")].copy()

# 1. Dates : conversions ratées ? écart entre raccordement et mise en service ?
print(ind[["dateRaccordement", "dateMiseEnService"]].isna().sum())
print((ind["dateMiseEnService"] - ind["dateRaccordement"]).dt.days.describe())

# 2. Effet des installations récentes sur la production
ind["annee_racc"] = ind["dateRaccordement"].dt.year
ind["kwh_par_kw"] = ind["energieAnnuelleGlissanteInjectee"] / ind["puisMaxInstallee"]
print(ind.groupby("annee_racc")["kwh_par_kw"].agg(["count", "median"]).tail(8))
print(ind["energieAnnuelleGlissanteInjectee"].isna().sum(),
      (ind["energieAnnuelleGlissanteInjectee"] == 0).sum())

# 3. Table centrale du projet : puissance cumulée (MW) par département et par année
annuel = (ind.groupby(["codeDepartement", "annee_racc"])["puisMaxInstallee"].sum()
             .unstack(fill_value=0)
             .reindex(columns=range(2007, 2027), fill_value=0)
             .cumsum(axis=1) / 1000)
print(annuel.round(1))

dateRaccordement     0
dateMiseEnService    3
dtype: int64
count    6319.000000
mean        0.411458
std        13.146436
min       -13.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       480.000000
dtype: float64
            count       median
annee_racc                    
2019          224  1331.116667
2020          288  1293.061616
2021          437  1305.739583
2022          471  1316.867470
2023          546  1244.038148
2024          847  1283.185525
2025         1095  1278.705882
2026          640   603.945789
173 29
annee_racc       2007  2008  2009  2010   2011   2012   2013   2014   2015  \
codeDepartement                                                              
04                0.0   0.0   8.9  38.4  155.5  175.0  177.3  187.4  207.0   
05                0.0   0.0   0.7   2.0    7.8   10.4   12.9   23.6   31.4   
07                0.0   0.0   0.2   3.0   12.7   20.4   25.1   44.4   45.6   
26                0.0   0.0   0.7  18.7   41.1   52

In [14]:
import pandas as pd
from pathlib import Path

RAW = Path("dataset/data raw")   # adapte si ton dossier s'appelle autrement
DEPS = ["26", "07", "38", "05", "04", "84"]

def lire(path):
    """Teste séparateur et encodage jusqu'à obtenir un tableau cohérent."""
    for sep in [";", ","]:
        for enc in ["utf-8", "latin-1"]:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, dtype=str, low_memory=False)
                if df.shape[1] > 5:
                    return df, sep, enc
            except Exception:
                continue
    raise ValueError(f"Lecture impossible : {path}")

infos, schemas, solaire = [], {}, {}
for p in sorted(RAW.glob("*.csv")):
    annee = int(p.name[:4])
    df, sep, enc = lire(p)
    schemas[annee] = set(df.columns)
    infos.append({"annee": annee, "fichier": p.name, "sep": sep, "enc": enc,
                  "lignes": len(df), "colonnes": df.shape[1]})
    # on ne garde en mémoire que le solaire de tes 6 départements, si les colonnes existent
    if {"codeDepartement", "filiere"} <= set(df.columns):
        solaire[annee] = df[df["codeDepartement"].isin(DEPS) & (df["filiere"] == "Solaire")].copy()
    else:
        print(f"{annee} : colonnes codeDepartement/filiere absentes")
    del df

pd.set_option("display.max_colwidth", None)
print(pd.DataFrame(infos).to_string(index=False))

ref = schemas[2026]
for a in sorted(schemas):
    print(f"\n{a} - manquantes vs 2026 :", sorted(ref - schemas[a]))
    print(f"{a} - en plus :", sorted(schemas[a] - ref))

 annee                                                      fichier sep   enc  lignes  colonnes
  2017 2017-installation-production-stockage-electricite-agrege.csv   ; utf-8   41237        47
  2018 2018-installation-production-stockage-electricite-agrege.csv   ; utf-8   44598        49
  2019 2019-installation-production-stockage-electricite-agrege.csv   ; utf-8   49043        47
  2020 2020-installation-production-stockage-electricite-agrege.csv   ; utf-8   54671        47
  2021 2021-installation-production-stockage-electricite-agrege.csv   ; utf-8   63410        49
  2022 2022-installation-production-stockage-electricite-agrege.csv   ; utf-8   72790        49
  2023 2023-installation-production-stockage-electricite-agrege.csv   ; utf-8   84944        50
  2024 2024-installation-production-stockage-electricite-agrege.csv   ; utf-8  101097        52
  2025 2025-installation-production-stockage-electricite-agrege.csv   ; utf-8  125930        51
  2026 2026-installation-production-stoc

In [15]:
def resume(s, annee):
    s = s.copy()
    for c in ["puisMaxInstallee", "energieAnnuelleGlissanteInjectee", "nbInstallations"]:
        if c in s:
            s[c] = pd.to_numeric(s[c].str.replace(",", ".", regex=False), errors="coerce")
    indiv = s[s["nbInstallations"] == 1] if "nbInstallations" in s else s
    return {
        "annee": annee,
        "lignes_solaire_6dep": len(s),
        "lignes_individuelles": len(indiv),
        "puissance_MW_indiv": round(indiv["puisMaxInstallee"].sum() / 1000, 1),
        "puissance_MW_agreg": round((s["puisMaxInstallee"].sum() - indiv["puisMaxInstallee"].sum()) / 1000, 1),
        "remplissage_dateRacc": round(s["dateRaccordement"].notna().mean(), 3) if "dateRaccordement" in s else None,
        "remplissage_injectee": round(s["energieAnnuelleGlissanteInjectee"].notna().mean(), 3) if "energieAnnuelleGlissanteInjectee" in s else None,
        "gestionnaires": sorted(indiv["gestionnaire"].dropna().unique()) if "gestionnaire" in indiv else None,
    }

tableau = pd.DataFrame([resume(s, a) for a, s in sorted(solaire.items())])
print(tableau.to_string(index=False))

# Valeurs de technologie par année (libellés stables ?)
for a, s in sorted(solaire.items()):
    if "technologie" in s:
        print(a, s["technologie"].fillna("NaN").value_counts().to_dict())

 annee  lignes_solaire_6dep  lignes_individuelles  puissance_MW_indiv  puissance_MW_agreg  remplissage_dateRacc  remplissage_injectee                                                           gestionnaires
  2017                 2537                  1438               354.0                21.9                   1.0                   NaN [Enedis, Energie développement services du Briançonnais, GreenAlp, RTE]
  2018                 2763                  1627               472.8                37.6                   1.0                   NaN [Enedis, Energie développement services du Briançonnais, GreenAlp, RTE]
  2019                 3040                  1847               608.7                46.6                   1.0                   NaN [Enedis, Energie développement services du Briançonnais, GreenAlp, RTE]
  2020                 3399                  2149               796.9               149.8                   1.0                 0.982 [Enedis, Energie développement services du

In [16]:
def puissance_indiv(s):
    s = s.copy()
    s["puis"] = pd.to_numeric(s["puisMaxInstallee"].str.replace(",", ".", regex=False), errors="coerce")
    if "technologie" in s:
        s = s[s["technologie"] == "Photovoltaïque"]
    if "nbInstallations" in s:
        s = s[pd.to_numeric(s["nbInstallations"], errors="coerce") == 1]
    elif "codeEICResourceObject" in s:
        s = s[s["codeEICResourceObject"].notna()]
    return s.groupby("codeDepartement")["puis"].sum() / 1000

snap = pd.DataFrame({a: puissance_indiv(s) for a, s in solaire.items() if a < 2026})
print(snap.round(1))                       # puissance vue par chaque fichier annuel
print((snap - annuel[list(snap.columns)]).round(1))   # écart avec ta reconstruction

                  2017   2018   2019   2020   2021   2022   2023   2024   2025
codeDepartement                                                               
04               205.2  207.2  231.2  265.8  354.7  365.8  382.1  431.6  486.9
05                34.5   52.8   62.1   77.6  103.5  134.9  149.5  209.3  229.3
07                16.5   30.8   52.5   78.7  106.4  115.9  129.0  139.5  153.6
26                31.2   61.5  100.5  139.6  194.2  234.6  257.3  303.2  367.1
38                13.4   28.7   35.2   61.7   79.2  111.4  167.4  220.6  306.5
84                50.6   91.7  127.2  173.4  190.0  211.9  233.9  278.1  316.8
                 2017  2018  2019  2020  2021  2022  2023  2024  2025
codeDepartement                                                      
04              -75.4 -84.5 -68.0 -62.7   3.2   6.6   3.0   2.0   9.5
05              -18.8 -18.1 -17.8  -6.7  -0.5   0.5  -0.6   0.9   1.3
07              -31.5 -30.3 -13.5  -0.3  -0.3  -5.7  -1.2   0.2   0.1
26              -5

In [17]:
def num(s, cols):
    s = s.copy()
    for c in cols:
        if c in s: s[c] = pd.to_numeric(s[c].str.replace(",", ".", regex=False), errors="coerce")
    return s

# TEST A : qualité de la puissance dans chaque fichier (lignes individuelles)
def qualite(s):
    i = num(s, ["puisMaxInstallee", "puisMaxRac", "nbInstallations"])
    i = i[i["nbInstallations"] == 1]
    return pd.Series({"lignes": len(i),
                      "puis_NaN": i["puisMaxInstallee"].isna().sum(),
                      "puis_<=0": (i["puisMaxInstallee"] <= 0).sum(),
                      "puis_mediane": i["puisMaxInstallee"].median(),
                      "puis_moyenne": i["puisMaxInstallee"].mean(),
                      "rac_somme_MW": i["puisMaxRac"].sum() / 1000})
print(pd.DataFrame({a: qualite(s) for a, s in solaire.items()}).T.round(1))

# TEST B : mêmes installations entre 2017 et 2026 ? (clé : code EIC)
a17 = num(solaire[2017], ["puisMaxInstallee", "nbInstallations"])
a26 = num(solaire[2026], ["puisMaxInstallee", "nbInstallations"])
i17 = a17[a17["nbInstallations"] == 1].drop_duplicates("codeEICResourceObject").set_index("codeEICResourceObject")
i26 = a26[a26["nbInstallations"] == 1].drop_duplicates("codeEICResourceObject").set_index("codeEICResourceObject")
communs = i17.index.intersection(i26.index)
print(len(i17), len(i26), len(communs))
cmp = pd.DataFrame({"p17": i17.loc[communs, "puisMaxInstallee"],
                    "p26": i26.loc[communs, "puisMaxInstallee"],
                    "commune": i26.loc[communs, "commune"]})
cmp["ecart"] = cmp["p26"] - cmp["p17"]
print((cmp["p17"] == cmp["p26"]).mean())            # part des installations à puissance identique
print(cmp["p17"].isna().sum(), (cmp["p17"] <= 0).sum())
print(cmp.sort_values("ecart", ascending=False).head(10))

# TEST C : production dans les fichiers 2017-2019
for a in [2017, 2018, 2019]:
    s = solaire[a]
    for c in ["energieAnnuelleInjectee", "energieAnnuelleProduite"]:
        print(a, c, round(s[c].notna().mean(), 3))

      lignes  puis_NaN  puis_<=0  puis_mediane  puis_moyenne  rac_somme_MW
2017  1438.0     389.0       0.0          95.0         337.5         593.7
2018  1627.0     316.0       0.0          98.8         360.6         713.2
2019  1847.0     204.0       0.0          99.0         370.5         789.4
2020  2149.0      87.0       0.0          99.0         386.5         898.4
2021  2604.0       3.0       0.0          99.0         395.3        1024.6
2022  3095.0       3.0       0.0          99.0         380.0        1167.6
2023  3672.0       3.0       0.0          99.0         359.5        1306.6
2024  4553.0       6.0       0.0         100.0         348.0        1542.6
2025  5682.0       0.0       0.0         100.0         327.4        1802.2
2026  6332.0       0.0       0.0         108.0         315.8        1945.5
1438 6332 1399
0.5396711937097927
366 0
                           p17       p26                      commune   ecart
codeEICResourceObject                                    

In [18]:
def to_num(x):
    return pd.to_numeric(x.str.replace(",", ".", regex=False), errors="coerce")

def to_date(x):
    d = pd.to_datetime(x, format="%d/%m/%Y", errors="coerce")
    manque = d.isna() & x.notna()
    if manque.any():
        d[manque] = pd.to_datetime(x[manque], errors="coerce")   # autre format éventuel
    return d

morceaux = []
for annee, s in sorted(solaire.items()):
    d = s[(to_num(s["nbInstallations"]) == 1) & (s["technologie"] == "Photovoltaïque")].copy()
    col_prod = ("energieAnnuelleGlissanteInjectee"
                if "energieAnnuelleGlissanteInjectee" in d else "energieAnnuelleInjectee")
    d["edition"] = annee
    d["puis_installee"] = to_num(d["puisMaxInstallee"])
    d["puis_rac"] = to_num(d["puisMaxRac"])
    d["injectee_kwh"] = to_num(d[col_prod])
    d["date_racc"] = to_date(d["dateRaccordement"])
    morceaux.append(d[["edition", "codeDepartement", "codeEICResourceObject", "commune",
                       "codeINSEECommune", "date_racc", "puis_installee", "puis_rac", "injectee_kwh"]])
panel = pd.concat(morceaux, ignore_index=True)

# Puissance en cascade, avec trace de la source
p26 = (panel[panel["edition"] == 2026].drop_duplicates("codeEICResourceObject")
       .set_index("codeEICResourceObject")["puis_installee"])
panel["p26"] = panel["codeEICResourceObject"].map(p26)
panel["puis_kw"] = panel["puis_installee"]
panel["source_puis"] = "installee"
for col, nom in [("puis_rac", "raccordement"), ("p26", "fichier_2026")]:
    m = panel["puis_kw"].isna() & panel[col].notna()
    panel.loc[m, "puis_kw"] = panel.loc[m, col]
    panel.loc[m, "source_puis"] = nom
panel.loc[panel["puis_kw"].isna(), "source_puis"] = "manquante"

# CONTRÔLE 1 : origine des puissances, par édition
print(panel.groupby(["edition", "source_puis"]).size().unstack(fill_value=0))

# CONTRÔLE 2 : total du panel vs reconstruction (annuel : table puissance cumulée, colonnes = années)
tot = panel.groupby("edition")["puis_kw"].sum().div(1000).round(1)
print(pd.DataFrame({"panel_MW": tot, "recon_MW": annuel.sum().reindex(tot.index).round(1)}))

# CONTRÔLE 3 : unité et période de la production (installations raccordées avant la période)
seuil = pd.to_datetime(panel["edition"].astype(str) + "-01-01")
seuil = seuil.where(panel["edition"] != 2026, pd.Timestamp("2025-09-01"))
mat = panel[panel["date_racc"] < seuil].copy()
mat["kwh_par_kw"] = mat["injectee_kwh"] / mat["puis_kw"]
print(mat.groupby("edition")["kwh_par_kw"].describe()[["count", "25%", "50%", "75%"]].round(0))

# CONTRÔLE 4 : date de la photographie de chaque édition
print(panel.groupby("edition")["date_racc"].max())

# Sauvegarde (après avoir regardé les contrôles ci-dessus)
# panel.to_csv("dataset/data clean/centrales_pv_6dep.csv", index=False, sep=";", encoding="utf-8")

source_puis  installee  raccordement
edition                             
2017              1048           370
2018              1311           312
2019              1643           203
2020              2062            86
2021              2600             3
2022              3090             3
2023              3669             3
2024              4547             6
2025              5680             0
2026              6322             0
         panel_MW  recon_MW
edition                    
2017        572.4     584.7
2018        700.4     706.2
2019        782.1     785.0
2020        893.4     894.9
2021       1032.9    1031.6
2022       1183.2    1184.6
2023       1327.8    1331.1
2024       1583.9    1581.1
2025       1860.2    1848.4
2026       1998.0    1998.0
          count     25%     50%     75%
edition                                
2017     1281.0  1238.0  1382.0  1513.0
2018     1419.0  1164.0  1289.0  1405.0
2019     1604.0  1229.0  1375.0  1509.0
2020     1830.0  119

In [19]:
# 1. Combien de lignes ont une date postérieure à la fin de leur édition ?
panel["fin_edition"] = panel["edition"].map(
    lambda a: pd.Timestamp("2026-08-31") if a == 2026 else pd.Timestamp(f"{a}-12-31"))
tard = panel[panel["date_racc"] > panel["fin_edition"]]
print(tard.groupby("edition").agg(lignes=("puis_kw", "size"),
                                  MW=("puis_kw", lambda x: round(x.sum() / 1000, 1))))
print(tard.sort_values("date_racc", ascending=False)
          .head(10)[["edition", "commune", "date_racc", "puis_kw", "injectee_kwh"]])
# Les mêmes installations se retrouvent-elles d'une édition à l'autre ?
print(tard.groupby("codeEICResourceObject")["edition"].agg(["min", "max", "count"]).head(10))

# 2. Doublons éventuels dans le panel
print(panel.duplicated(["edition", "codeEICResourceObject"]).sum())

# 3. Synthèse par département et par édition (2017-2025)
synth = (panel[panel["edition"] <= 2025]
         .groupby(["codeDepartement", "edition"])
         .agg(centrales=("puis_kw", "size"),
              puissance_MW=("puis_kw", lambda x: x.sum() / 1000),
              injectee_GWh=("injectee_kwh", lambda x: x.sum() / 1e6),
              part_prod_renseignee=("injectee_kwh", lambda x: x.notna().mean()))
         .round(3).reset_index())
print(synth.pivot(index="codeDepartement", columns="edition", values="puissance_MW").round(1))
print(synth.pivot(index="codeDepartement", columns="edition", values="injectee_GWh").round(1))
print(synth.pivot(index="codeDepartement", columns="edition", values="part_prod_renseignee"))

# 4. Sauvegarde dans data clean (après avoir lu les sorties ci-dessus)
# panel.to_csv("dataset/data clean/centrales_pv_6dep.csv", index=False, sep=";", encoding="utf-8")
# synth.to_csv("dataset/data clean/synthese_dep_edition.csv", index=False, sep=";", encoding="utf-8")

         lignes   MW
edition             
2017          2  0.2
2018          7  0.8
2019          3  0.3
2020         14  3.1
2021         10  1.3
2022          5  3.9
2023          7  1.2
       edition                commune  date_racc  puis_kw  injectee_kwh
14908     2023       Le Poët-Sigillat 2024-07-19     99.0           NaN
15492     2023              Romagnieu 2024-06-28    250.0           NaN
16333     2023       La Tour-d'Aigues 2024-04-29     70.0           NaN
14948     2023       La Roche-de-Glun 2024-04-29    200.0           NaN
14735     2023             Montélimar 2024-04-25    160.0           NaN
14803     2023            Valherbasse 2024-03-21    165.0           NaN
14398     2023  Châteauneuf-sur-Isère 2024-03-08    230.0           NaN
9874      2022               Montclar 2023-11-07   3440.0           NaN
12366     2022                Grillon 2023-06-08     99.0           NaN
8841      2021        Bougé-Chambalud 2023-03-15    110.0           NaN
                   

In [20]:
p = panel[panel["edition"] <= 2025].copy()
# "mature" = raccordée au moins un an avant le début de l'année de l'édition (choix prudent)
p["mature"] = p["date_racc"] < pd.to_datetime((p["edition"] - 1).astype(str) + "-01-01")

# 1. Production manquante : installations récentes ou vrai trou de données ?
print(p.groupby(["edition", "mature"])["injectee_kwh"].agg(lambda x: x.isna().mean()).unstack().round(3))

# 2. Rendement des installations matures (kWh/kW), par département et édition
m = p[p["mature"] & (p["injectee_kwh"] > 0)]
g = m.groupby(["codeDepartement", "edition"]).agg(prod=("injectee_kwh", "sum"), puis=("puis_kw", "sum"))
print((g["prod"] / g["puis"]).unstack().round(0))

# 3. Part de la puissance qui a pu produire une année complète
p["puis_mature"] = p["puis_kw"].where(p["mature"], 0)
g2 = p.groupby(["codeDepartement", "edition"]).agg(tot=("puis_kw", "sum"), mat=("puis_mature", "sum"))
print((g2["mat"] / g2["tot"]).unstack().round(2))

# 4. Sauvegardes dans data clean
cols = ["edition", "codeDepartement", "codeEICResourceObject", "commune", "codeINSEECommune",
        "date_racc", "puis_kw", "source_puis", "injectee_kwh"]
panel[cols].to_csv("dataset/data clean/centrales_pv_6dep.csv", index=False, sep=";", encoding="utf-8")
synth.to_csv("dataset/data clean/synthese_dep_edition.csv", index=False, sep=";", encoding="utf-8")

mature   False  True 
edition              
2017     0.013  0.009
2018     0.019  0.011
2019     0.022  0.010
2020     0.093  0.007
2021     0.087  0.010
2022     0.074  0.014
2023     0.079  0.014
2024     0.057  0.023
2025     0.029  0.025
edition            2017    2018    2019    2020    2021    2022    2023  \
codeDepartement                                                           
04               1601.0  1432.0  1691.0  1548.0  1515.0  1551.0  1539.0   
05               1476.0  1327.0  1525.0  1543.0  1492.0  1529.0  1484.0   
07               1346.0  1290.0  1363.0  1347.0  1328.0  1360.0  1303.0   
26               1307.0  1287.0  1362.0  1335.0  1298.0  1392.0  1314.0   
38               1220.0  1154.0  1196.0  1328.0  1230.0  1348.0  1205.0   
84               1405.0  1321.0  1403.0  1375.0  1349.0  1392.0  1420.0   

edition            2024    2025  
codeDepartement                  
04               1392.0  1256.0  
05               1241.0  1305.0  
07               1228

In [21]:
m = p[p["mature"] & p["injectee_kwh"].gt(0)].copy()
m["rend"] = m["injectee_kwh"] / m["puis_kw"]

# Test 1 : la baisse vient-elle de toute la distribution ou d'une queue de mauvais producteurs ?
print(m.groupby("edition")["rend"].quantile([.1, .25, .5, .75, .9]).unstack().round(0))

# Test 2 : mêmes installations matures suivies de 2022 à 2025 (supprime l'effet de composition)
eds = [2022, 2023, 2024, 2025]
sub = p[p["edition"].isin(eds) & p["injectee_kwh"].gt(0)]
base = sub[(sub["edition"] == 2022) & sub["mature"]]["codeEICResourceObject"]
bal = sub[sub["codeEICResourceObject"].isin(base)]
nb = bal.groupby("codeEICResourceObject")["edition"].nunique()
bal = bal[bal["codeEICResourceObject"].isin(nb[nb == len(eds)].index)]
print(bal["codeEICResourceObject"].nunique(), "installations suivies")
g = bal.groupby(["codeDepartement", "edition"]).agg(prod=("injectee_kwh", "sum"), puis=("puis_kw", "sum"))
print((g["prod"] / g["puis"]).unstack().round(0))

# Rendement de référence par département (installations matures, 2017-2025) et productible normalisé
r = m.groupby("codeDepartement").agg(prod=("injectee_kwh", "sum"), puis=("puis_kw", "sum"))
rend_ref = (r["prod"] / r["puis"]).round(0)
print(rend_ref)
synth["rend_ref_kwh_kw"] = synth["codeDepartement"].map(rend_ref)
synth["productible_GWh"] = synth["puissance_MW"] * synth["rend_ref_kwh_kw"] / 1000

           0.10    0.25    0.50    0.75    0.90
edition                                        
2017     1080.0  1225.0  1369.0  1510.0  1634.0
2018     1043.0  1156.0  1284.0  1401.0  1499.0
2019     1087.0  1220.0  1361.0  1500.0  1616.0
2020     1029.0  1184.0  1320.0  1454.0  1575.0
2021      976.0  1150.0  1299.0  1443.0  1574.0
2022      984.0  1180.0  1333.0  1478.0  1603.0
2023      951.0  1159.0  1312.0  1462.0  1595.0
2024      909.0  1079.0  1237.0  1373.0  1511.0
2025      875.0  1109.0  1270.0  1414.0  1552.0
2078 installations suivies
edition            2022    2023    2024    2025
codeDepartement                                
04               1551.0  1531.0  1383.0  1243.0
05               1530.0  1481.0  1300.0  1350.0
07               1360.0  1231.0  1187.0  1200.0
26               1393.0  1324.0  1205.0  1229.0
38               1348.0  1199.0  1088.0  1113.0
84               1392.0  1426.0  1281.0  1240.0
codeDepartement
04    1492.0
05    1410.0
07    1292.0
26    

In [22]:

# SUPERFICIES DEPARTEMENTALES
# Codes conservés en texte, comme dans le panel.

superficies = pd.DataFrame({
    "codeDepartement": ["04", "05", "07", "26", "38", "84"],
    "departement": [
        "Alpes-de-Haute-Provence",
        "Hautes-Alpes",
        "Ardèche",
        "Drôme",
        "Isère",
        "Vaucluse"
    ],
    "superficie_km2": [
        6925, 5697, 5529, 6530, 7431, 3567
    ]
})

# Contrôle : une seule ligne par département
assert superficies["codeDepartement"].is_unique
assert set(superficies["codeDepartement"]) == set(
    synth["codeDepartement"].astype(str).unique()
)

# Jointure avec la synthèse départementale
synth["codeDepartement"] = synth["codeDepartement"].astype(str)
synth = synth.merge(
    superficies,
    on="codeDepartement",
    how="left",
    validate="many_to_one"
)

# Indicateurs de densité photovoltaïque
# puissance_MW est en MW ; productible_GWh est en GWh.
synth["puissance_MW_km2"] = (
    synth["puissance_MW"] / synth["superficie_km2"]
)
synth["productible_GWh_km2"] = (
    synth["productible_GWh"] / synth["superficie_km2"]
)

# Vérifications
print("Superficies manquantes :", synth["superficie_km2"].isna().sum())
print(
    synth.loc[synth["edition"] == 2025, [
        "codeDepartement", "departement", "superficie_km2",
        "puissance_MW", "puissance_MW_km2",
        "productible_GWh", "productible_GWh_km2"
    ]].round(4).sort_values("codeDepartement")
)

# Export
synth.to_csv(
    "dataset/data clean/synthese_dep_edition.csv",
    index=False,
    sep=";",
    encoding="utf-8"
)


Superficies manquantes : 0
   codeDepartement              departement  superficie_km2  puissance_MW  \
8               04  Alpes-de-Haute-Provence            6925       486.922   
17              05             Hautes-Alpes            5697       229.284   
26              07                  Ardèche            5529       153.606   
35              26                    Drôme            6530       367.102   
44              38                    Isère            7431       306.511   
53              84                 Vaucluse            3567       316.822   

    puissance_MW_km2  productible_GWh  productible_GWh_km2  
8             0.0703         726.4876               0.1049  
17            0.0402         323.2904               0.0567  
26            0.0278         198.4590               0.0359  
35            0.0562         469.1564               0.0718  
44            0.0412         362.9090               0.0488  
53            0.0888         425.1751               0.1192  


In [23]:

# Correction de la superficie des Hautes-Alpes (05)
# sans refaire la jointure

synth.loc[
    synth["codeDepartement"].astype(str) == "05",
    "superficie_km2"
] = 5549

# Recalcul des indicateurs de densité
synth["puissance_MW_km2"] = (
    synth["puissance_MW"] / synth["superficie_km2"]
)

synth["productible_GWh_km2"] = (
    synth["productible_GWh"] / synth["superficie_km2"]
)

# Vérification
print(
    synth.loc[
        (synth["codeDepartement"].astype(str) == "05")
        & (synth["edition"] == 2025),
        [
            "codeDepartement",
            "departement",
            "superficie_km2",
            "puissance_MW",
            "puissance_MW_km2",
            "productible_GWh",
            "productible_GWh_km2"
        ]
    ].round(4)
)


   codeDepartement   departement  superficie_km2  puissance_MW  \
17              05  Hautes-Alpes            5549       229.284   

    puissance_MW_km2  productible_GWh  productible_GWh_km2  
17            0.0413         323.2904               0.0583  


In [24]:

synth.to_csv(
    "dataset/data clean/synthese_dep_edition.csv",
    index=False,
    sep=";",
    encoding="utf-8"
)

print("Fichier de synthèse corrigé et enregistré.")


Fichier de synthèse corrigé et enregistré.


In [25]:
from pathlib import Path
import pandas as pd

fichier = Path("dataset/data clean/synthese_dep_edition.csv")

assert fichier.exists(), f"Fichier introuvable : {fichier}"

synthese_verif = pd.read_csv(
    fichier,
    sep=";",
    encoding="utf-8",
    dtype={"codeDepartement": str}
)

# Check required columns
colonnes_requises = [
    "codeDepartement",
    "edition",
    "departement",
    "superficie_km2",
    "puissance_MW",
    "productible_GWh"
]

assert all(
    col in synthese_verif.columns
    for col in colonnes_requises
), "Une ou plusieurs colonnes requises sont absentes."

# Check department codes
codes_attendus = {"04", "05", "07", "26", "38", "84"}
codes_obtenus = set(synthese_verif["codeDepartement"].dropna())

assert codes_obtenus == codes_attendus, (
    f"Codes trouvés : {sorted(codes_obtenus)}"
)

# Check duplicate department-year records
doublons = synthese_verif.duplicated(
    subset=["codeDepartement", "edition"]
).sum()

print("File found:", fichier)
print("Number of rows:", len(synthese_verif))
print("Departments:", sorted(codes_obtenus))
print("Years:", sorted(synthese_verif["edition"].unique()))
print("Duplicate department-year records:", doublons)

display(synthese_verif.head())

File found: dataset\data clean\synthese_dep_edition.csv
Number of rows: 54
Departments: ['04', '05', '07', '26', '38', '84']
Years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Duplicate department-year records: 0


,codeDepartement,edition,centrales,puissance_MW,injectee_GWh,part_prod_renseignee,rend_ref_kwh_kw,productible_GWh,departement,superficie_km2,puissance_MW_km2,productible_GWh_km2
0,04,2017,258,288.969,393.932,1.000,1492.0,431.141748,Alpes-de-Haute-Provence,6925,0.041728,0.062259
1,04,2018,272,289.045,426.951,0.996,1492.0,431.255140,Alpes-de-Haute-Provence,6925,0.041739,0.062275
2,04,2019,293,295.993,478.490,1.000,1492.0,441.621556,Alpes-de-Haute-Provence,6925,0.042743,0.063772
3,04,2020,331,327.912,475.911,0.982,1492.0,489.244704,Alpes-de-Haute-Provence,6925,0.047352,0.070649
4,04,2021,395,354.653,519.491,0.970,1492.0,529.142276,Alpes-de-Haute-Provence,6925,0.051213,0.076410


In [26]:
# Check missing values in key indicators
colonnes_a_verifier = [
    "puissance_MW",
    "productible_GWh",
    "superficie_km2",
    "puissance_MW_km2",
    "productible_GWh_km2"
]

print("Missing values per column:")
display(synthese_verif[colonnes_a_verifier].isna().sum())

# Check that departmental areas are positive
assert (synthese_verif["superficie_km2"] > 0).all()

# Recalculate the two ratios independently
puissance_calculee = (
    synthese_verif["puissance_MW"]
    / synthese_verif["superficie_km2"]
)

productible_calcule = (
    synthese_verif["productible_GWh"]
    / synthese_verif["superficie_km2"]
)

# Compare recalculated ratios with exported values
ecart_puissance = (
    synthese_verif["puissance_MW_km2"] - puissance_calculee
).abs()

ecart_productible = (
    synthese_verif["productible_GWh_km2"] - productible_calcule
).abs()

print("Maximum absolute difference in power ratio:",
      ecart_puissance.max())

print("Maximum absolute difference in yield ratio:",
      ecart_productible.max())

print("Power ratio consistent:", ecart_puissance.max() < 1e-6)
print("Yield ratio consistent:", ecart_productible.max() < 1e-6)

Missing values per column:


puissance_MW           0
productible_GWh        0
superficie_km2         0
puissance_MW_km2       0
productible_GWh_km2    0
dtype: int64

Maximum absolute difference in power ratio: 9.8879238130678e-17
Maximum absolute difference in yield ratio: 1.0408340855860843e-16
Power ratio consistent: True
Yield ratio consistent: True
